In [7]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

In [8]:
train_data = pd.read_json('train.jsonl', lines=True)
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
train_data = json_normalize(train_data.to_dict(orient='records'))
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

In [9]:
def extract_full_text(tweet):
    text = tweet['text']
    if not pd.isna(tweet['extended_tweet.full_text']):
        text = tweet['extended_tweet.full_text']
    return text

train_data['full_text'] = train_data.apply(lambda tweet: extract_full_text(tweet), axis=1)
kaggle_data['full_text'] = kaggle_data.apply(lambda tweet: extract_full_text(tweet), axis=1)

In [10]:
import re

def extract_source_platform(source_html):
    """Extract platform name from Twitter source HTML"""
    if pd.isna(source_html):
        return 'unknown'
    
    # Extract text between > and < in the anchor tag
    match = re.search(r'>([^<]+)<', source_html)
    if match:
        platform = match.group(1).strip()
        
        # Clean up common patterns
        platform = platform.replace('Twitter for ', '')
        platform = platform.replace('Twitter Web ', '')
        
        return platform.lower()
    
    # If no match, try to extract from URL
    url_patterns = {
        'iphone': 'iphone',
        'ipad': 'ipad',
        'android': 'android',
        'mobile.twitter.com': 'mobile web',
        'twitter.com': 'web',
        'publicize.wp.com': 'wordpress',
        'tweetdeck': 'tweetdeck',
        'instagram': 'instagram',
        'foursquare': 'foursquare'
    }
    
    source_lower = source_html.lower()
    for pattern, name in url_patterns.items():
        if pattern in source_lower:
            return name
    
    return 'other'

# Apply the extraction
train_data['source'] = train_data['source'].apply(extract_source_platform)
kaggle_data['source'] = kaggle_data['source'].apply(extract_source_platform)

# Check the results
print(train_data['source'].value_counts())
print(kaggle_data['source'].value_counts())

source
iphone            49536
app               45072
android           44903
ipad               3520
tweetdeck          2558
                  ...  
polyalert-live        1
publer.io             1
tic mag               1
mytwt_mycron          1
fcm live tweet        1
Name: count, Length: 408, dtype: int64
source
iphone                  33530
app                     29723
android                 29714
ipad                     2445
tweetdeck                1737
                        ...  
lamariejeanne media         1
zut-dev                     1
itfb                        1
andstatus-opensource        1
echofon                     1
Name: count, Length: 278, dtype: int64


In [11]:
columns_to_keep = [
    'created_at',
    'source',
    'is_quote_status',
    'challenge_id',
    'label',
    'entities.urls',
    'entities.user_mentions',
    'entities.symbols',
    'user.profile_image_url_https',
    'user.listed_count',
    'user.profile_background_image_url',
    'user.favourites_count',
    'user.description',
    'user.created_at',
    'user.is_translator',
    'user.profile_background_image_url_https',
    'user.profile_link_color',
    'user.translator_type',
    'user.geo_enabled',
    'user.profile_background_color',
    'user.profile_sidebar_border_color',
    'user.profile_text_color',
    'user.profile_image_url',
    'user.profile_background_tile',
    'user.profile_banner_url',
    'user.statuses_count',
    'user.profile_use_background_image',
    'user.default_profile',
    'user.location',
    'user.profile_sidebar_fill_color',
    'extended_tweet.entities.urls',
    'extended_tweet.entities.user_mentions',
    'full_text',
]


In [12]:
train_data = train_data[[col for col in columns_to_keep if col in train_data.columns]]
kaggle_data = kaggle_data[[col for col in columns_to_keep if col in kaggle_data.columns]]

In [13]:
train_data.to_pickle('train_filtered.pkl')
kaggle_data.to_pickle('kaggle_filtered.pkl')


print("Data saved successfully!")

Data saved successfully!


In [14]:
# Load later:
T = pd.read_pickle('train_filtered.pkl')
K = pd.read_pickle('kaggle_filtered.pkl')

In [21]:
print((T.columns))

Index(['created_at', 'source', 'is_quote_status', 'challenge_id', 'label',
       'entities.urls', 'entities.user_mentions', 'entities.symbols',
       'user.profile_image_url_https', 'user.listed_count',
       'user.profile_background_image_url', 'user.favourites_count',
       'user.description', 'user.created_at', 'user.is_translator',
       'user.profile_background_image_url_https', 'user.profile_link_color',
       'user.translator_type', 'user.geo_enabled',
       'user.profile_background_color', 'user.profile_sidebar_border_color',
       'user.profile_text_color', 'user.profile_image_url',
       'user.profile_background_tile', 'user.profile_banner_url',
       'user.statuses_count', 'user.profile_use_background_image',
       'user.default_profile', 'user.location',
       'user.profile_sidebar_fill_color', 'extended_tweet.entities.urls',
       'extended_tweet.entities.user_mentions', 'full_text'],
      dtype='object')
